https://machinelearningmastery.com/example-applications-of-text-embedding/

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Define a corpus of articles (title and content)
articles = [
  {
    "title": "Understanding Deep Learning",
    "content": ("Deep learning is a subset of machine learning where artificial neural networks, "
                "algorithms inspired by the human brain, learn from large amounts of data.")
  },
  {
    "title": "Introduction to Natural Language Processing",
    "content": ("Natural Language Processing (NLP) is a field of AI that gives machines the "
                "ability to read, understand, and derive meaning from human languages.")
  },
  {
    "title": "The Future of Computer Vision",
    "content": ("Computer vision is an interdisciplinary field that deals with how computers can "
                "gain high-level understanding from digital images or videos.")
  },
  {
    "title": "Reinforcement Learning Explained",
    "content": ("Reinforcement learning is an area of machine learning concerned with how "
                "software agents ought to take actions in an environment so as to maximize some "
                "notion of cumulative reward.")
  },
  {
    "title": "Neural Networks and Their Applications",
    "content": ("Neural networks are a set of algorithms, modeled loosely after the human brain, "
                "that are designed to recognize patterns in data.")
  }
]

model = SentenceTransformer("all-MiniLM-L6-v2")

def create_article_embeddings(articles, model):
    """create embeddings for articles"""
    texts = [f"{article['title']}. {article['content']}" for article in articles]
    embeddings = model.encode(texts)
    return embeddings

def get_recommendations(article_id, articles, embeddings, top_n=2):
    """get recommendations for a given article ID based on cosine similarity"""
    similarities = cosine_similarity([embeddings[article_id]], embeddings)[0]
    similar_indices = np.argsort(similarities)[::-1][1:top_n+1]
    return [articles[idx] for idx in similar_indices]

# Create embeddings for all articles, and get recommendation for first article
embeddings = create_article_embeddings(articles, model)
recommendations = get_recommendations(0, articles, embeddings)

# Print the recommendations
print(f'Recommendations for "{articles[0]["title"]}":')
for i, rec in enumerate(recommendations):
    print(f"{i+1}. {rec['title']}")

Recommendations for "Understanding Deep Learning":
1. Neural Networks and Their Applications
2. Reinforcement Learning Explained


In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

corpus = [
  {
    "language": "English",
    "text": ("Machine learning is a field of study that gives computers the ability to learn "
             "without being explicitly programmed.")
  },
  {
    "language": "Spanish",
    "text": ("El aprendizaje automático es un campo de estudio que da a las computadoras la "
             "capacidad de aprender sin ser programadas explícitamente.")
  },
  {
    "language": "French",
    "text": ("L'apprentissage automatique est un domaine d'étude qui donne aux ordinateurs "
             "la capacité d'apprendre sans être explicitement programmés.")
  },
  {
    "language": "German",
    "text": ("Maschinelles Lernen ist ein Studienbereich, der Computern die Fähigkeit gibt, "
             "zu lernen, ohne explizit programmiert zu werden.")
  },
  {
    "language": "Italian",
    "text": ("Il machine learning è un campo di studio che conferisce ai computer la capacità "
             "di apprendere senza essere esplicitamente programmati.")
  },
  {
    "language": "English",
    "text": ("Natural language processing is a subfield of linguistics, computer science, "
             "and artificial intelligence.")
  },
  {
    "language": "English",
    "text": ("Computer vision is an interdisciplinary field that deals with how computers can "
             "gain high-level understanding from digital images or videos.")
  }
]

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Generate embeddings for the corpus
texts = [doc["text"] for doc in corpus]
embeddings = model.encode(texts)

# Define a query in English and generate an embedding
query = "What is machine learning?"
query_embedding = model.encode(query)

# Sort the embeddings of the corpus by descending similarity
similarities = cosine_similarity([query_embedding], embeddings)[0]
ranked_indices = np.argsort(similarities)[::-1]

# Print ranked results
print(f"Query: {query}\n")
for i, idx in enumerate(ranked_indices[:3]):  # Show top 3 results
    print(f"{i+1}. [{corpus[idx]['language']}] {corpus[idx]['text']} (Similarity: {similarities[idx]:.4f})")

Query: What is machine learning?

1. [Italian] Il machine learning è un campo di studio che conferisce ai computer la capacità di apprendere senza essere esplicitamente programmati. (Similarity: 0.8129)
2. [English] Machine learning is a field of study that gives computers the ability to learn without being explicitly programmed. (Similarity: 0.7788)
3. [French] L'apprentissage automatique est un domaine d'étude qui donne aux ordinateurs la capacité d'apprendre sans être explicitement programmés. (Similarity: 0.7470)


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

articles = [
    # Business articles
    {"text": "The stock market reached a new high today, with technology stocks leading the gains.", "category": "Business"},
    {"text": "The government announced a new tax policy that will affect small businesses.", "category": "Business"},
    {"text": "The central bank has decided to keep interest rates unchanged.", "category": "Business"},
    {"text": "Quarterly earnings reports exceeded expectations for most Fortune 500 companies.", "category": "Business"},
    {"text": "Inflation rates have decreased for the third consecutive month.", "category": "Business"},
    {"text": "The merger between two major corporations has been approved by regulators.", "category": "Business"},
    {"text": "Unemployment rates have fallen to a five-year low according to new data.", "category": "Business"},
    {"text": "The cryptocurrency market experienced significant volatility this week.", "category": "Business"},

    # Health articles
    {"text": "A new study shows that regular exercise can reduce the risk of heart disease.", "category": "Health"},
    {"text": "A clinical trial for a new cancer treatment has shown promising results.", "category": "Health"},
    {"text": "A balanced diet and regular sleep are essential for maintaining good health.", "category": "Health"},
    {"text": "Medical researchers have identified a new gene linked to Alzheimer's disease.", "category": "Health"},
    {"text": "The WHO has issued new guidelines for managing diabetes in elderly patients.", "category": "Health"},
    {"text": "A new technique for early detection of breast cancer has been developed.", "category": "Health"},
    {"text": "Studies show that mindfulness meditation can help reduce stress and anxiety.", "category": "Health"},
    {"text": "Public health officials warn of a potential flu outbreak this winter season.", "category": "Health"},

    # Technology articles
    {"text": "The latest smartphone from Apple features a better camera and longer battery life.", "category": "Technology"},
    {"text": "The new electric car from Tesla has a range of over 400 miles.", "category": "Technology"},
    {"text": "The latest update to the operating system includes new security features.", "category": "Technology"},
    {"text": "A new artificial intelligence system can detect diseases from medical images.", "category": "Technology"},
    {"text": "The tech company unveiled its new virtual reality headset at the annual conference.", "category": "Technology"},
    {"text": "Researchers have developed a quantum computer that can solve complex problems.", "category": "Technology"},
    {"text": "The new social media platform has gained millions of users in just a few months.", "category": "Technology"},
    {"text": "Cybersecurity experts warn of a new type of malware targeting smart home devices.", "category": "Technology"},

    # Science articles
    {"text": "Scientists have discovered a new species of frog in the Amazon rainforest.", "category": "Science"},
    {"text": "Astronomers have observed a supernova in a distant galaxy.", "category": "Science"},
    {"text": "Researchers have developed a new method for measuring ocean temperatures.", "category": "Science"},
    {"text": "A fossil discovery suggests that dinosaurs may have been warm-blooded.", "category": "Science"},
    {"text": "Climate scientists report that Arctic ice is melting at an unprecedented rate.", "category": "Science"},
    {"text": "Physicists have confirmed the existence of a new subatomic particle.", "category": "Science"},
    {"text": "A study of coral reefs shows signs of recovery in protected marine areas.", "category": "Science"},
    {"text": "Biologists have sequenced the genome of an endangered species of tiger.", "category": "Science"}
]


# Prepare data for classification training
model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [article["text"] for article in articles]
X = model.encode(texts)
y = [article["category"] for article in articles]

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data into training and testing sets with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# Train a logistic regression classifier with regularization
classifier = LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000)
classifier.fit(X_train, y_train)

# Evaluate the classifier
y_pred = classifier.predict(X_test)
print(classification_report(y_test, y_pred))

# Classify new articles
new_articles = [
    "The company reported a 20% increase in quarterly profits.",
    "A new vaccine has been approved for use against the flu.",
    "The new laptop features a faster processor and more memory.",
    "The Mars rover has sent back new images of the planet\"s surface."
]
new_embeddings = model.encode(new_articles)
new_embeddings_scaled = scaler.transform(new_embeddings)
new_predictions = classifier.predict(new_embeddings_scaled)
for article, prediction in zip(new_articles, new_predictions):
    
    print(f"Article: {article}\nPredicted Category: {prediction}\n")

              precision    recall  f1-score   support

    Business       1.00      1.00      1.00         2
      Health       0.50      1.00      0.67         1
     Science       1.00      1.00      1.00         2
  Technology       1.00      0.50      0.67         2

    accuracy                           0.86         7
   macro avg       0.88      0.88      0.83         7
weighted avg       0.93      0.86      0.86         7

Article: The company reported a 20% increase in quarterly profits.
Predicted Category: Business

Article: A new vaccine has been approved for use against the flu.
Predicted Category: Health

Article: The new laptop features a faster processor and more memory.
Predicted Category: Technology

Article: The Mars rover has sent back new images of the planet"s surface.
Predicted Category: Science

